In [ ]:
%load_ext autoreload
%autoreload 2
import jax.numpy as jnp
import jax_sn

import jax_sbgeom as jsb
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh
import pyvista as pv
from typing import Iterable
from jax_sn.domain.cross_sections import CrossSectionData, CrossSectionsIndexed
import numpy as onp 
import matplotlib.pyplot as plt
import equinox as eqx

In [ ]:
vmec_file = ["/home/tbogaarts/stellarator_paper/base_data/vmecs/helias5_vmec.nc4"][0]
fs_base = jsb.flux_surfaces.FluxSurfaceNormalExtendedNoPhi.from_hdf5(vmec_file)

In [ ]:
d_edge_norm = 1.5
flux_surface_extended = jsb.flux_surfaces.create_extended_flux_surface_d_interp_equal_arclength(fs_base, d_edge_norm, 70, 80, 100)

In [ ]:
dim = 2
degree = 4
tn_order = 3
n_elem_regions = (10, 1, 1, 10, 5, 1, 10, 2)
groups_list = (14.1e6, 10e6, 3e6, 1e6, 1e4, 300, 10,0)

d_regions_base = (1.2, 0.002, 0.025, 0.5, 0.385, 0.06, 0.2, 0.06)
l_order = 4
source_kwargs = {
    "n0"    : 2.4e20,
    "Ti0"   : 15.2,
    "nalpha": 0.35,
    "Tialpha": 1.2,
}
element = jax_sn.solution_domain.BasixLagrangianSimplex(degree, dim)

In [ ]:
from breeding_blanket_setup import _get_hcpb_data_openmc
new_bb_data_dict, new_aux_data_dict, tally_result_dict = _get_hcpb_data_openmc(d_regions_base, l_order, 1e5, groups_list)

In [ ]:
phi               = jnp.pi/25
n_theta          = 40

In [ ]:
from breeding_blanket_setup import setup_mgop
MGOP_EA, solution_domain_ea, source_all_angles_ea = setup_mgop(d_regions_base, phi, n_theta, n_elem_regions, tn_order, flux_surface_extended, source_kwargs, new_bb_data_dict, element, normalisation_distance = d_edge_norm)

In [ ]:
from jax_sn.operators.multi_group_operator.lower_triangular_solver import MultiGroupLowerTriangularBlockSolver
from jax_sn.operators.multi_group_operator.multi_group_operator_imp import _solve_multigroup_operator_rhs_scan
from breeding_blanket_setup import line_interpolate, plot_domain_with_lines_grid
import lineax as lx
mgsolver = MultiGroupLowerTriangularBlockSolver(lx.BiCGStab(rtol = 1e-10, atol=1e-10, max_steps = 250))


In [ ]:
solution_total_ea = _solve_multigroup_operator_rhs_scan(MGOP_EA, source_all_angles_ea, mgsolver)

In [ ]:
from jax_sbgeom.jax_utils.utils import pchip

In [ ]:
class BZShield2DParametrisation(eqx.Module):
    d_bz     : jnp.ndarray
    d_shield : jnp.ndarray

class BZShield2DSettings(eqx.Module):
    n_theta : int
    
 

def _get_values(vertices_base : jnp.ndarray, ):
    import jax_sbgeom as jsb
    s_power_sampling = 2
    regions_dim = [(sz, res) for sz, res in zip(d_regions_base, n_elem_regions)]
    cumsum_nodes  = jnp.concatenate([jnp.array([0]), jnp.cumsum(jnp.array([sz for sz, res in regions_dim]))])        

    s_base = [jnp.linspace(cumsum_nodes[0], cumsum_nodes[1], n_elem_regions[0], endpoint=False)**s_power_sampling]
    s_from_edge = [jnp.linspace(cumsum_nodes[i], cumsum_nodes[i+1], n_elem_regions[i], endpoint=False) for i in range(1, len(regions_dim))] + [jnp.array([cumsum_nodes[-1]])]
    if normalise_distance is not None:
        assert isinstance(normalise_distance, float), f"normalise_distance should be a float but is {type(normalise_distance)}"
        assert normalise_distance > cumsum_nodes[-1] - 1, f"normalise_distance ({normalise_distance}) should be larger than the total size of the blanket ({cumsum_nodes[-1] - 1})"        
        s_array_base = jnp.concatenate(s_base + s_from_edge)
        s_array = jnp.where(s_array_base < 1.0, s_array_base, 1.0 + (s_array_base - 1.0) / (normalise_distance ))
    else:
        s_array = jnp.concatenate(s_base + s_from_edge)

    mesh_pol = jsb.flux_surfaces.flux_surface_meshing.mesh_poloidal_plane(flux_surface_extended,s_array, phi, n_theta )
    print(mesh_pol[0])


_get_values(d_regions_base, n_elem_regions, flux_surface_extended, n_theta, phi, source_kwargs, normalise_distance = d_edge_norm)